In [ ]:
import torch
import math
import numpy as np
import matplotlib.pyplot as plt

# import evaluate_samples function from /work/work_fran/sampling/experiments/utils/evaluate.py
# import sys
# sys.path.append("/work/work_fran/sampling/experiments")
# from utils.evaluate import evaluate_samples

import os
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# device = torch.device("cpu")
print("Using device:", DEVICE)

In [ ]:
COLORS_DICT = {
    "gt": "black",
    "progressive_interpolation_sde" : "cornflowerblue",
    "parallel_tempering" : "indianred", 
    "smc" : "seagreen",
    "digs" : "darkorange",
    "mala" : "turquoise",
    "hmc" : "violet",
}

NAMES_TO_SHOW = {
    "gt": "Ground Truth",
    "parallel_tempering" : "NRPT", 
    "progressive_interpolation_sde" : "CDS (ours)",
    "smc" : "OASMC",
    "digs" : "DiGS",
    "mala" : "MALA",
    "hmc" : "HMC",
    "gmnu_2_40": "GMNU-2",
    "gm_2_40": "GM-2",
    "gmnu_16_40": "GMNU-16",
    "gm_16_40": "GM-16",
    "lennard_jones_13": "LJ-13",
    "lennard_jones_55": "LJ-55",
    "aldp_vacuum": "ALDP",
    "mlp_posterior": "BNN",
    "test/w2_data": "$W_2$",
    "test/mmd_energy": "MMD",
    "test/tvd_energy": "TVD",
    "test/w2_energy": "$W_2$ density",
    "test/relative_mae": "Relative MAE",
    "test/w2_data_equivariant": "$W_2$",
    "test/rel_mae_equivariant": "Relative MAE",
    "test/kl_div_ramachandran": "Ramachandran KL",
    "test/test_nll": "Test NLL",
    "num_density_evals": "Density evaluations",
}


In [ ]:
from sampling.targets import LennardJones


TARGET = LennardJones(
    n_particles=55,
    data_path="/home/fran/work_fran/sampling/experiments/data/all_data_LJ55-1000-part1.npy"
).to(DEVICE)


EPS = 1e-6
N_SAMPLES = 10000
gt_samples = TARGET.sample(N_SAMPLES).to(DEVICE)
gt_energies = -TARGET.log_prob(gt_samples)

# RESULTS_PATH = "/home/fran/work_fran/sampling/experiments/notebooks/results"
RESULTS_PATH = "/home/fran/work_fran/sampling/experiments/notebooks/results_arxiv"
SAVE_PATH = "/home/fran/work_fran/sampling/experiments/figures/"

In [ ]:
def plot_results(ax, method_name, samples, alpha=1.0):
    energies = -TARGET.log_prob(samples.to(DEVICE))
    energies = energies.detach().cpu().numpy()

    ax.hist(
        energies,
        bins=100,
        density=True,
        histtype="step",
        color=COLORS_DICT[method_name],
        label=NAMES_TO_SHOW[method_name],
        linewidth=2,
        alpha=alpha,
    )

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
plot_results(ax, "gt", gt_samples)

In [ ]:
def build_schedule(min_val, max_val, n_steps):
    r = (max_val / min_val) ** (1.0 / (n_steps - 1))
    schedule = min_val * (r ** torch.arange(n_steps))
    return schedule

In [ ]:
from sampling.samplers import PT, ProgressiveInterpolationSDE, MALA, HMC, DiGS, SMC
from sampling.kernels import MALAKernel


COMPILE = True
N_STEPS = 10000

init_samples_path = "/home/fran/work_fran/sampling/experiments/data/lennard_jones_55_minenergy.pt"
# x0 = torch.zeros((N_SAMPLES, TARGET.dim)).to(DEVICE)
init_samples = torch.load(init_samples_path)
x0 = init_samples[0, :].unsqueeze(0).repeat(N_SAMPLES, 1).to(DEVICE)

def build_sampler(name):
    if name == "parallel_tempering":
        
        n_replicas = 3
        beta_schedule = build_schedule(0.8, 1.0, n_replicas).to(DEVICE)

        kernel = MALAKernel()

        sampler = PT(
            target=TARGET,
            kernel=kernel,
            beta_schedule=beta_schedule,
            step_size=0.001,
            verbose=True,
            compile=COMPILE
        ).to(DEVICE)
    elif name == "progressive_interpolation_sde":
        
        n_replicas = 3
        int_steps = 100
        jump_steps = math.ceil(N_STEPS  - int_steps) // n_replicas

        time_schedule = build_schedule(0.3, 1.0, int_steps+2).to(DEVICE)
        base_noise_var = 0.001
        noise_schedule = torch.full_like(time_schedule, base_noise_var)
        jump_beta_schedule = build_schedule(0.8, 1.0, n_replicas).to(DEVICE)

        sampler = ProgressiveInterpolationSDE(
            target=TARGET,
            time_schedule=time_schedule,
            noise_schedule=noise_schedule,
            corrector_mode="mala",
            corrector_steps=1,
            corrector_adaptation_rate=0.0,
            corrector_step_size=0.001,
            jump_ref_std=1.0,
            jump_steps=jump_steps,
            jump_beta_schedule=jump_beta_schedule,
            compile=COMPILE,
            verbose=True
        ).to(DEVICE)
    elif name == "smc":
        
        n_particles = 20
        n_steps_smc = N_STEPS // n_particles
        beta_schedule = build_schedule(0.8, 1.0, n_steps_smc).to(DEVICE)
        kernel = MALAKernel()

        sampler = SMC(
            target=TARGET,
            kernel=kernel,
            beta_schedule=beta_schedule,
            n_particles=n_particles,
            step_size=0.001,
            verbose=True,
            compile=COMPILE
        ).to(DEVICE)
    
    elif name == "digs":

        n_noise_levels = 1
        n_denoising_steps = 1
        n_steps_digs = N_STEPS // (1 + n_denoising_steps)
        n_gibbs_sweeps = n_steps_digs // n_noise_levels

        alpha_shedule = torch.linspace(0.1, 0.9, n_noise_levels)
        sigma_schedule = torch.sqrt(1.0 - alpha_shedule ** 2)

        sampler = DiGS(
            target=TARGET,
            denoising_kernel=MALAKernel(),
            alpha_schedule=alpha_shedule.to(DEVICE),
            sigma_schedule=sigma_schedule.to(DEVICE),
            n_denoising_steps=n_denoising_steps,
            n_gibbs_sweeps=n_gibbs_sweeps,
            denoising_step_size=0.001,
            compile=COMPILE,
            verbose=True,
        )
    
    elif name == "mala":
        sampler = MALA(
            target=TARGET,
            step_size=0.0001,
            verbose=True,
            compile=COMPILE,
            adaptation_rate=0.01
        ).to(DEVICE)
    elif name == "hmc":
        sampler = HMC(
            target=TARGET,
            step_size=0.0001,
            n_leapfrog_steps=3,
            verbose=True,
            compile=COMPILE,
            adaptation_rate=0.01
        ).to(DEVICE)
    else:
        raise ValueError(f"Unknown sampler: {name}")
    
    return sampler
    

def run_sampler(sampler, name):
    if name == "parallel_tempering":
        n_steps = N_STEPS // sampler.beta_schedule.shape[0]
        samples = sampler(
            x0=x0,
            n_steps=n_steps
        )
        samples = samples[:, -1, :]
    elif name == "progressive_interpolation_sde":
        z = x0.clone()
        samples = sampler(
            x0=x0,
            z=z
        )
    elif name == "smc":
        samples = sampler(
            x0=x0
        )
    elif name == "digs":
        samples = sampler(
            x0=x0,
        )
    elif name == "mala":
        n_steps = N_STEPS
        samples = sampler(
            x0=x0,
            n_steps=n_steps
        )
    elif name == "hmc":
        n_steps = N_STEPS // sampler.n_leapfrog_steps
        samples = sampler(
            x0=x0,
            n_steps=n_steps
        )
    else:
        raise ValueError(f"Unknown sampler: {name}")

    return samples


In [ ]:


samplers_to_run = [
    # "progressive_interpolation_sde",
    # "parallel_tempering",
    "smc",
    # "digs",
    # "hmc",
    # "mala"
]


for sampler_name in samplers_to_run:
    print(f"Running sampler: {sampler_name}")

    sampler = build_sampler(sampler_name)
    samples = run_sampler(sampler, sampler_name)
    torch.save(
        samples,
        f"{RESULTS_PATH}/samples_lj55_{sampler_name}.pt",
    )

In [ ]:
plt.rcParams.update({'font.size': 16})

fig, ax = plt.subplots(1, 1, figsize=(5, 5))

samplers_to_show = [
    "mala",
    "hmc",
    # "digs",
    # "smc",
    "parallel_tempering",
    "progressive_interpolation_sde",
]

for sampler_name in samplers_to_show:
    try:
        samples = torch.load(
            f"{RESULTS_PATH}/samples_lj55_{sampler_name}.pt",
        )
    except FileNotFoundError:
        print(f"Samples for {sampler_name} not found, skipping.")
        continue
    plot_results(ax, sampler_name, samples)
    # ax.set_title(sampler_name)
plot_results(ax, "gt", gt_samples, alpha=0.7)
ax.set_xlabel("Potential Energy")
# ax.set_ylabel("Density")
ax.grid(True, linestyle='--', alpha=0.5)
ax.set_xlim([None, -280.0])
ax.legend(
    fontsize=12,
    loc='upper right',
    frameon=False,
)

plt.tight_layout()
plt.savefig(SAVE_PATH + f"samples_lj55_reduced.pdf", dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
plt.rcParams.update({'font.size': 16})

fig, ax = plt.subplots(1, 1, figsize=(5, 5))

samplers_to_show = [
    "mala",
    "hmc",
    # "digs",
    "smc",
    "parallel_tempering",
    "progressive_interpolation_sde",
]

for sampler_name in samplers_to_show:
    print(f"Plotting sampler: {sampler_name}")
    samples = torch.load(
        f"{RESULTS_PATH}/samples_lj55_{sampler_name}.pt",
    )
    plot_results(ax, sampler_name, samples)
    # ax.set_title(sampler_name)
plot_results(ax, "gt", gt_samples, alpha=0.7)
ax.set_xlabel("Potential Energy")
# ax.set_ylabel("Density")
ax.grid(True, linestyle='--', alpha=0.5)
ax.set_xlim([None, -280])
ax.legend(
    fontsize=12,
    loc='upper left',
    frameon=False,
)

plt.tight_layout()
plt.savefig(SAVE_PATH + f"samples_lj55_all.pdf", dpi=300, bbox_inches='tight')
plt.show()